# Урок 7.1. Зачем несколько агентов

Модуль 7 «Мульти-агентные системы» курса «LLM для Python-разработчиков».

В уроке 3.5 мы построили агента: одна модель, цикл, инструменты. В этом модуле —
следующий уровень: **несколько LLM-ролей, работающих над одной задачей**. Звучит
модно, стоит дорого, помогает не всегда — поэтому первый урок не про «как»,
а про «когда»: живьём измерим, где одна модель ломается, а где команда ролей
реально чинит результат.

🎯 **Цели урока:**

- Увидеть вживую, как «все роли в одном промпте» разваливаются на бюджете и смешении ролей.
- Выучить три базовых паттерна: pipeline, supervisor, debate — и когда какой.
- Прогнать debate на спорном тикете и увидеть, как вторая пара глаз меняет решение.
- Посчитать честную цену команды агентов в токенах и секундах.
- Сформулировать правило: когда один агент лучше трёх.

**Что понадобится:** Ollama с qwen2.5:3b. Маленькая модель здесь не компромисс,
а часть сюжета: ограничения 3B в агентных сценариях — сквозная тема модуля.

## Где мы в курсе

| Модуль | Что умеем |
|---|---|
| 1–2 | промпты, few-shot, эмбеддинги, свой поиск |
| 3 | RAG, eval, guardrails, **агент с инструментами (3.5)**, LangGraph (3.6) |
| 4 | безопасность, мониторинг, reranking, права |
| 5 | локальный прод: Ollama, GGUF, прокси |
| 6 | стиль в весах: LoRA, Modelfile, приёмка |
| **7 (вы здесь)** | **команды агентов: роли, оркестрация, цена** |

Сквозная задача модуля — та же вселенная «Пингвин.Хост» 🐧: команда из
**аналитика, разработчика и ревьюера** решает небольшие инженерные задачи
хостинга. В этом уроке — почему для этого вообще нужна команда; в 7.2 построим
оркестратор на чистом Python; в 7.3 соберём то же на CrewAI; в 7.4 сравним
фреймворки.

Определимся с термином. **Агент** в этом модуле — это вызов LLM со своей ролью
(системным промптом), своим бюджетом и, возможно, своими инструментами.
**Мульти-агентная система** — несколько таких вызовов плюс код, который решает,
кто говорит следующим и что передать дальше, — **оркестратор**.

## 1. Отправная точка: все роли в одном промпте

Инженерная задача из бэклога «Пингвин.Хост»: панель управления показывает лимиты
дисков строками («1.5 ГБ»), а биллингу нужны байты. Нужна функция `parse_size`.

Хорошая практика и без всяких LLM выглядит так: аналитик выписывает требования
и крайние случаи, разработчик пишет код, ревьюер ищет проблемы. Первый порыв —
попросить модель сыграть все три роли в одном ответе. Попробуем и посмотрим
внимательно, что получится.

In [ ]:
import json
import time

import httpx

OLLAMA_URL = "http://127.0.0.1:11434"   # 127.0.0.1, не localhost - урок 3.2
MODEL = "qwen2.5:3b"

def chat(messages: list[dict], num_predict: int = 500, temperature: float = 0):
    """Один вызов /api/chat; возвращает (ответ, телеметрия) - привычка урока 5.5."""
    started = time.perf_counter()
    response = httpx.post(f"{OLLAMA_URL}/api/chat", json={
        "model": MODEL, "messages": messages, "stream": False,
        "options": {"temperature": temperature, "num_predict": num_predict},
    }, timeout=300)
    response.raise_for_status()
    data = response.json()
    meta = {"prompt_tokens": data.get("prompt_eval_count", 0),
            "answer_tokens": data.get("eval_count", 0),
            "done_reason": data.get("done_reason"),
            "seconds": round(time.perf_counter() - started, 1)}
    return data["message"]["content"], meta

TASK = """Напиши функцию Python parse_size(text), которая переводит человекочитаемый
размер диска в число байтов. Примеры: "512 МБ" -> 536870912, "1.5 ГБ" -> 1610612736,
"2 ТБ" -> 2199023255552. Поддержи КБ/МБ/ГБ/ТБ, дробные числа, пробел необязателен."""

ONE_PROMPT = f"""Ты выполняешь три роли по очереди в одном ответе.

ЗАДАЧА: {TASK}

Сначала как АНАЛИТИК выпиши все требования и крайние случаи.
Затем как РАЗРАБОТЧИК напиши полную функцию.
Затем как РЕВЬЮЕР найди минимум 3 проблемы в своём коде и исправь их.
В конце выведи финальную версию функции."""

try:
    answer, meta = chat([{"role": "user", "content": ONE_PROMPT}], num_predict=900)
    print(f"[{meta['seconds']} c | ответ {meta['answer_tokens']} ток. | "
          f"done_reason={meta['done_reason']}]\n")
    print("...", answer[-700:])
except httpx.HTTPError as exc:
    print(f"[пропущено: Ollama недоступна - {exc.__class__.__name__}. "
          f"Запустите её и выполните ollama pull {MODEL}]")

Смотрите на хвост моего прогона (у вас детали будут другими — GPU-недетерминизм
из урока 3.3, — но болезни воспроизводятся стабильно):

1. **`done_reason=length`** — ответ упёрся в бюджет из 900 токенов и оборвался
   **посреди финальной функции** (на `.lower` без скобок). Три роли в одном
   ответе — это длинно, а бюджет один на всех. Поднять `num_predict`? Тогда один
   такой запрос начнёт стоить как небольшая команда агентов — и всё равно без
   гарантий.
2. **Само-ревью беззубое.** Присмотритесь к «найденным проблемам»: пункт 3
   предлагает «добавить удаление пробела после единицы измерения» — при том, что
   первая же строка кода уже удаляет **все** пробелы. Модель не анализирует свой
   код, а генерирует правдоподобно звучащие замечания.
3. **Дистанции нет.** Модель проверяет текст, который сама только что написала:
   те же веса, тот же контекст, те же слепые пятна (эффект знаком по уроку 3.3 —
   судья снисходителен к «своему»). Настоящие баги — например, единицы, которые
   регэксп не поймает после lower() — остались не замечены.

Диагноз: роли конфликтуют за один контекст и один бюджет. Лекарство напрашивается —
дать каждой роли **отдельный вызов**: свой промпт, свой бюджет, свой фокус.
Это и есть мульти-агентная система, в самом честном смысле слова.

## 2. Три базовых паттерна

Всё многообразие мульти-агентных архитектур собирается из трёх кирпичей:

**Pipeline (конвейер)** — роли по очереди, выход одной = вход следующей:

```
 задача ──▶ [аналитик] ──спека──▶ [разработчик] ──код──▶ [ревьюер] ──▶ результат
```

**Supervisor (диспетчер)** — одна LLM-роль разбирает задачу и раздаёт подзадачи
исполнителям, потом собирает результат:

```
                    ┌──▶ [исполнитель A] ──┐
 задача ─▶ [супервизор]                    ├──▶ [супервизор] ─▶ результат
                    └──▶ [исполнитель B] ──┘
```

**Debate (спор)** — несколько мнений об одном и том же + судья:

```
            ┌──▶ [мнение 1] ──┐
 вопрос ────┤                 ├──▶ [судья] ──▶ решение
            └──▶ [критик]  ───┘
```

| Паттерн | Когда | Вызовов LLM | Главный риск |
|---|---|---|---|
| pipeline | этапы задачи разные по сути (анализ → код → проверка) | N ролей (+ циклы доработки) | ошибка ранней роли едет по конвейеру дальше |
| supervisor | подзадачи заранее неизвестны, набор исполнителей разный | 2 + K подзадач | супервизор ошибается в декомпозиции — всё зря |
| debate | одно решение, высокая цена ошибки, есть правила для судьи | мнения + 1 | дорого; спорщики могут дружно ошибаться |

Заметьте: **pipeline мы уже строили** — RAG из модуля 3 это конвейер
`ретривер → генератор → судья`, просто ретривер там не LLM. Мульти-агентность —
не новая магия, а продолжение той же инженерии.

## 3. Debate вживую: тикет-крик

Помните ловушку из бенчмарка урока 5.2 — тикет «ВСЁ СЛОМАЛОСЬ!!!»? По регламенту
поддержки это **low**: в тикете нет ни одного факта (что лежит? у скольких?),
сначала надо запросить детали. Но эмоциональное давление — то, на что LLM
покупается лучше всего.

Сцена из четырёх актов: **классификатор** решает, **критик** ищет слабое место,
**наивный судья** выносит вердикт по мнениям сторон — а потом тот же судья, но
с **чеклистом** (жёстким форматом, который заставляет сначала выписать факты,
и только потом решать). Правила у всех одни и те же — интересно, кто их удержит.

In [ ]:
TICKET = "ВСЁ СЛОМАЛОСЬ!!! НИЧЕГО НЕ РАБОТАЕТ!!! СРОЧНО!!!"

RULES = """Правила срочности:
high - подтверждённая авария (прод лежит, данные теряются),
medium - функция сломана, но есть обход,
low - вопрос или нет деталей для оценки."""

try:
    total = {"prompt": 0, "answer": 0}

    def step(prompt: str, num_predict: int = 300) -> str:
        answer, meta = chat([{"role": "user", "content": prompt}], num_predict)
        total["prompt"] += meta["prompt_tokens"]
        total["answer"] += meta["answer_tokens"]
        return answer

    verdict1 = step(f"""Классифицируй тикет поддержки по срочности: low, medium или high.
{RULES}
Тикет: "{TICKET}"
Ответь JSON: {{"severity": "...", "reason": "..."}}""")
    print("КЛАССИФИКАТОР:", verdict1[:180], "\n")

    critique = step(f"""Ты — придирчивый старший инженер поддержки. Коллега классифицировал тикет.
Тикет: "{TICKET}"
Вердикт коллеги: {verdict1}
Найди слабое место в его рассуждении. В тикете есть ли КОНКРЕТНЫЕ факты об аварии
(что именно лежит? сколько пользователей?)? Ответь коротко: согласен или нет и почему.""")
    print("КРИТИК:", critique[:300], "\n")

    naive_judge = step(f"""Ты — судья. Тикет: "{TICKET}"
{RULES}
Мнение классификатора: {verdict1}
Мнение критика: {critique}
Прими финальное решение строго по правилам.
Ответь JSON: {{"severity": "...", "reason": "..."}}""")
    print("НАИВНЫЙ СУДЬЯ:", naive_judge[:180], "\n")

    checklist_judge = step(f"""Ты — судья по тикетам поддержки. Тикет: "{TICKET}"
Мнение критика: {critique}

Заполни чеклист строго по тикету (не выдумывай):
1. named_failures: какие КОНКРЕТНЫЕ сервисы/функции названы сломанными (список, может быть пустым)
2. impact_facts: какие факты о масштабе указаны (список, может быть пустым)
3. severity: если фактов нет - "low" (нет деталей для оценки); high только если названы факты аварии.

Ответь JSON: {{"named_failures": [...], "impact_facts": [...], "severity": "...", "reason": "..."}}""")
    print("СУДЬЯ С ЧЕКЛИСТОМ:", checklist_judge[:340])
    print(f"\nЦена: 4 вызова LLM, {total['prompt']} ток. промптов, "
          f"{total['answer']} ток. ответов")
except httpx.HTTPError as exc:
    print(f"[пропущено: Ollama недоступна - {exc.__class__.__name__}]")

Мой прогон — спектакль в четырёх актах, и он поучительнее, чем «debate всех спас»:

- **Классификатор: `high`**, причём в `reason` он дословно переписал формулировку
  правила — «прод лежит, данные теряются». В тикете ничего этого **нет**: модель
  не процитировала факты, а сочинила их под выбранный ответ. Крик победил правила.
- **Критик: «нет конкретных фактов»** — вопрос с узким фокусом вернул рассуждение
  к регламенту. Заодно критик вставил в русский текст китайские иероглифы
  (受到影响 — «затронуты») — поздоровайтесь с ограничением маленьких qwen, мы видели
  его в уроке 6.3: **выходы агентов нужно валидировать кодом**, об этом урок 7.2.
- **Наивный судья: `high`** — дословно скопировал вердикт классификатора, вместе
  с выдуманным reason, проигнорировав критика. Вот это важно: **судья — тоже
  маленькая модель**, и «прими решение по мнениям сторон» для неё слишком
  свободная задача. Добавить агента — не значит добавить надёжности.
- **Судья с чеклистом: `low`.** Тот же самый судья, но формат заставил его сначала
  выписать факты из тикета (оба списка пустые), и только потом выбрать severity.
  Это CoT из урока 1.7, упакованный в жёсткую структуру: маленькой модели нельзя
  доверить *свободное* взвешивание, но можно — *заполнение формы*.

Отсюда два вывода модуля сразу. Дистанция работает (критик увидел то, мимо чего
проехал автор), но **паттерн сильнее ролей**: слабое звено debate — судья, и
чинится он не заменой модели, а жёстким форматом с проверяемыми полями.

И цена: одно решение — 4 вызова и ~1000 токенов против одного вызова и ~200.
За крик в тикете — может, и оправданно; за каждый из 5000 тикетов в день
(профиль нагрузки из урока 5.1) — считайте сами.

## 4. Честная арифметика команды

Агенты общаются **последовательно** (выход одного — вход другого), поэтому
складываются и токены, и секунды:

| Схема | Вызовов | Латентность | Токены | Комментарий |
|---|---|---|---|---|
| один агент | 1 | 1× | 1× | база |
| pipeline из 3 ролей | 3 | ~3× | ~3–4× | + каждый следующий читает выход предыдущего |
| debate (мнение+критик+судья) | 3 | ~3× | ~4× | судья читает всех |
| supervisor с K подзадачами | 2 + K | ~(2+K)× | растёт с K | + супервизор читает все результаты |
| цикл доработки (dev↔review) | +2 за итерацию | +2× за итерацию | копится история | лимит итераций обязателен (урок 7.2) |

Правила выживания при такой арифметике:

- **Каждой роли — только нужный ей контекст.** Не пересылайте всю историю всем:
  разработчику нужна спека, а не крик клиента; судье — вердикты, а не вся переписка.
- **Каждой роли — свой `num_predict`**: критику хватит 200 токенов, разработчику
  нужно 600. Это ровно та проблема, о которую разбился «один промпт на три роли».
- **Выход каждой роли проверяется кодом** до передачи дальше — иначе ошибка
  (или иероглифы) едет по конвейеру как по маслу.

## 5. Когда один агент лучше трёх

Правило старта — то же, что в уроке 3.5 («начинай с конвейера, агент — когда
маршрут неизвестен»), поднятое на уровень выше:

> **Начинайте с одного агента. Вторую роль добавляйте, когда eval показал класс
> ошибок, который ловится «второй парой глаз», — и этот выигрыш окупает ×3 цену.**

Один агент лучше, когда:

- задача — один связный шаг (суммаризация, классификация без ловушек, ответ по RAG);
- жёсткий бюджет латентности: каждый агент в цепочке — плюс секунды (наш debate
  шёл ~20 секунд против ~5 у одиночного вызова);
- у ролей нет **проверяемых выходов**: если некому и нечем оценить вклад критика,
  вы платите ×3 за ощущение надёжности;
- проблему решает ступень пониже из лестницы курса: лучший промпт (1.7), RAG (3),
  жёсткая грамматика (5.4), дообучение (6). Мульти-агентность — верх лестницы,
  а не первый инструмент.

А команда агентов оправдана, когда: этапы задачи требуют **разного фокуса**
(анализ/код/проверка), цена ошибки высока и есть правила для судьи, или подзадачи
заранее неизвестны (supervisor). Всё это — при проверяемых выходах ролей.

## ⚠️ Частые ошибки

1. **Мульти-агентность ради моды.** «Сделаем 7 агентов как в статье» — сначала
   один агент + eval; роли добавляют по одной, под конкретный класс ошибок.
2. **Роли без проверяемого выхода.** Если вклад «критика» нельзя измерить, его
   нельзя и оправдать. Метрика конца конвейера + проверка выхода каждой роли.
3. **Общий чат всех со всеми.** N агентов в одном чате = N² прочтений чужих
   сообщений и лавина токенов. Оркестратор передаёт только нужное (урок 7.2).
4. **Циклы без лимитов.** «Дорабатывай, пока ревьюер не одобрит» — вечный цикл
   на маленькой модели гарантирован. Лимит итераций — как max_steps в 3.5.
5. **Само-ревью вместо ревью.** Проверять должен другой вызов с другим контекстом
   (а лучше — код с тестами); модель к своему тексту снисходительна.
6. **Оценка по красоте диалога.** Переписка агентов выглядит убедительно даже
   когда результат мусор. Судите по метрике финального артефакта.

## Упражнения

1. **Цена supervisor.** Задача разбивается на K=4 подзадачи; супервизор тратит
   ~300 токенов на декомпозицию и ~400 на сборку, каждый исполнитель — ~250
   промпта и ~350 ответа. Посчитайте суммарные токены и сравните с одиночным
   вызовом (~250+600). Во сколько раз дороже?

2. **Почините демо из раздела 1 двумя вызовами.** Первый вызов — аналитик:
   только требования и крайние случаи, JSON-списком. Второй — разработчик:
   пишет функцию по этому JSON (не по исходной задаче!). Сравните с «одним
   промптом»: дописан ли код до конца (`done_reason`), учтены ли «МБ» и «mb».

3. **Debate из вашей практики.** Придумайте решение с высокой ценой ошибки
   (деплой в пятницу? возврат денег? бан пользователя?), сформулируйте правила
   для судьи и роли спорщиков. Что будет проверяемым выходом каждой роли?

In [ ]:
# Решение 2: две роли - два вызова, у каждого свой бюджет и свой фокус
try:
    spec_json, meta1 = chat([{"role": "user", "content": f"""Ты — аналитик. НЕ пиши код.
ЗАДАЧА: {TASK}
Выпиши 4-6 требований и 3-4 крайних случая, коротко.
Ответь строго JSON: {{"requirements": ["..."], "edge_cases": ["..."]}}"""}],
                            num_predict=450)
    print(f"АНАЛИТИК [{meta1['answer_tokens']} ток., done={meta1['done_reason']}]:")
    print(spec_json, "\n")

    code_answer, meta2 = chat([{"role": "user", "content": f"""Ты — Python-разработчик.
Напиши функцию parse_size(text) -> int строго по этой спецификации аналитика:
{spec_json}
Верни только код одним блоком, без пояснений."""}], num_predict=600)
    print(f"РАЗРАБОТЧИК [{meta2['answer_tokens']} ток., done={meta2['done_reason']}]:")
    print(code_answer)
except httpx.HTTPError as exc:
    print(f"[пропущено: Ollama недоступна - {exc.__class__.__name__}]")

**Разбор решения 2** (мой прогон): оба вызова закончились `done_reason=stop` —
никто не упёрся в бюджет, потому что бюджет у каждого свой. Связь ролей работает:
аналитик сам добавил требование «пустая строка → 0» (заметьте: **выдумал** его,
в задаче такого не было — аналитики-LLM расширяют ТЗ незаметно), а разработчик
честно перенёс его в код первой проверкой.

Но присмотритесь к регэкспу: `^(\d+(\.\d+)?)([КМГТ]?)$` не пропускает ни пробел,
ни букву «Б» — на эталонном входе `"512 МБ"` этот аккуратный на вид код упадёт
с ValueError. И **никто в команде этого не заметил**, потому что никто не
*запускает* код: обе роли работают с текстом. Судья с чеклистом был хорош для
тикета, а лучший судья для кода — интерпретатор с тестами. Именно так мы устроим
приёмку в оркестраторе урока 7.2.

**Решение 1.** Супервизор: 300+400 ответных + чтение результатов; исполнители:
4×(250+350). Итого промптов ~250+4×250+~1400(сборка читает всё) и ответов
~300+4×350+400 ≈ 2100. Против ~850 у одиночного вызова — **в 3–4 раза дороже**
по токенам и в ~6 раз по вызовам. Если исполнители не делают ничего
специализированного — это переплата за красоту.

**Решение 3** — проверьте себя: если для судьи не нашлось письменных правил,
дебату нечего применять — сначала регламент, потом агенты (иначе вы построили
генератор уверенных случайных решений).

## Мини-квиз

**1. Чем мульти-агентная система отличается от агента с инструментами из урока 3.5?**

<details><summary>Ответ</summary>

Агент 3.5 — один вызов LLM в цикле, который сам решает, какие инструменты дёргать.
Мульти-агентная система — несколько LLM-вызовов с разными ролями и промптами,
между которыми код-оркестратор передаёт сообщения. Инструменты могут быть у обоих;
разница — в специализации ролей и внешней координации.
</details>

**2. Почему «сам найди 3 проблемы в своём коде» работает хуже, чем отдельный вызов-ревьюер?**

<details><summary>Ответ</summary>

У само-ревью нет дистанции: тот же контекст, те же слепые пятна, и всё это — в одном
бюджете токенов. Отдельный вызов читает код «свежими глазами» без груза черновиков
и получает собственный num_predict. (А ещё лучше ревьюера-LLM работают тесты — урок 7.2.)
</details>

**3. Критик верно заметил «нет фактов», но наивный судья всё равно вынес high. Почему — и как это починили?**

<details><summary>Ответ</summary>

Судья — та же маленькая модель: свободная задача «взвесь мнения сторон» для неё
слишком широкая, и она скопировала первый попавшийся уверенный JSON. Починили
жёстким форматом: чеклист заставил сначала выписать факты из тикета (списки
оказались пустыми) и только потом выбрать severity по правилу. Не новая модель,
а структура рассуждения.
</details>

**4. В каком случае supervisor-паттерн — переплата?**

<details><summary>Ответ</summary>

Когда декомпозиция известна заранее и всегда одинакова — тогда это обычный pipeline,
и LLM-супервизор не нужен: маршрут можно зашить кодом (дешевле и предсказуемее).
Supervisor оправдан, когда набор подзадач зависит от входа.
</details>

**5. Команда из 3 агентов даёт на eval +2% качества к одиночному агенту. Внедрять?**

<details><summary>Ответ</summary>

Считать: ×3 латентность и ×3–4 токены за +2%. Для дешёвого фонового пайплайна —
может быть; для чат-бота под нагрузкой — почти наверняка нет. Решение принимает
не «качество любой ценой», а метрика вместе с ценой (уроки 1.3 и 5.1 — та же логика).
</details>

**6. Какой минимальный механизм обязателен в любом цикле «разработчик ↔ ревьюер»?**

<details><summary>Ответ</summary>

Лимит итераций (и критерий выхода, проверяемый кодом). Без него две маленькие
модели могут вежливо дорабатывать друг друга бесконечно — как агент без max_steps
в уроке 3.5.
</details>

## 🎓 Итоги

- «Все роли в одном промпте» разваливается живьём: общий бюджет
  (`done_reason=length` посреди финального кода), смешение ролей, беззубое
  само-ревью.
- Три кирпича мульти-агентности: **pipeline** (разные этапы), **supervisor**
  (неизвестная декомпозиция), **debate** (важное решение + правила для судьи).
- Debate на тикете-крике: классификатор поверил крику (`high` с выдуманным
  reason), критик увидел «нет фактов», наивный судья скопировал `high`, и только
  судья с чеклистом вынес `low`. Мораль: **паттерн сильнее ролей** — слабое звено
  чинится жёстким форматом с проверяемыми полями, а не добавлением голов.
- Цена честная: ×3–4 вызова, ×4–5 токенов, ×3 латентность. Критик вставил
  иероглифы — выходы агентов будем валидировать кодом.
- Правило: начинайте с одного агента; роли добавляйте под измеренный класс ошибок.

**Дальше — урок 7.2:** строим оркестратор на чистом Python: сообщения, общая
память, команда «аналитик → разработчик → ревьюер» с настоящими тестами в роли
судьи и лимитом итераций.

## 📚 Что почитать

- [Anthropic: Building effective agents](https://www.anthropic.com/research/building-effective-agents) — паттерны и то же правило «проще — лучше»
- [LangGraph: multi-agent architectures](https://langchain-ai.github.io/langgraph/concepts/multi_agent/) — supervisor и команды на графах
- [ChatDev](https://arxiv.org/abs/2307.07924) и [MetaGPT](https://arxiv.org/abs/2308.00352) — статьи про «софтверные компании из агентов», с которых начался хайп
